# 02 - Data Cleaning

This notebook cleans and prepares the raw data for merging and feature engineering.

**Input:** Raw data from `data/raw/elap733/` and `data/raw/nba_api/`

**Output:** Cleaned datasets in `data/processed/`:
- `injury_history_by_player_season.csv` - Aggregated injury data (TARGET VARIABLE)
- `player_stats_combined.csv` - Combined player stats across all seasons
- `player_bio_combined.csv` - Combined player bio data
- `tracking_stats_combined.csv` - Combined tracking data (2013+)
- `team_back_to_backs.csv` - Back-to-back game flags
- `player_id_mapping.csv` - Mapping between player names and NBA IDs

In [1]:
import sys
sys.path.append('..')

import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# For fuzzy matching
from difflib import SequenceMatcher

from src.config import (
    FIRST_SEASON, LAST_SEASON,
    TRACKING_DATA_START,
    RAW_ELAP_DIR, RAW_NBA_API_DIR,
    PROCESSED_DIR
)

from src.utils import (
    MANUAL_NAME_MAPPINGS,
    EXCLUDE_FROM_INJURY_DATA
)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# Ensure output directory exists
Path(f'../{PROCESSED_DIR}').mkdir(parents=True, exist_ok=True)

print(f"Season range: {FIRST_SEASON} to {LAST_SEASON}")
print(f"Tracking data from: {TRACKING_DATA_START}")
print(f"Output directory: {PROCESSED_DIR}")
print(f"Manual name mappings: {len(MANUAL_NAME_MAPPINGS)}")
print(f"Excluded names (coaches/staff): {len(EXCLUDE_FROM_INJURY_DATA)}")

Season range: 2010 to 2019
Tracking data from: 2013
Output directory: data/processed
Manual name mappings: 48
Excluded names (coaches/staff): 14


---
## Section 1: Load All Raw Data

In [2]:
# Load elap733 data
print("Loading elap733 data...")
print("="*60)

df_missed_games_raw = pd.read_csv(f'../{RAW_ELAP_DIR}/missed_games_2010_2019.csv', index_col=0)
print(f"missed_games: {df_missed_games_raw.shape}")

df_injury_transactions = pd.read_csv(f'../{RAW_ELAP_DIR}/injury_transactions_2010_2019.csv', index_col=0)
print(f"injury_transactions: {df_injury_transactions.shape}")

df_elap_stats = pd.read_csv(f'../{RAW_ELAP_DIR}/player_stats_1994_2019.csv', index_col=0)
print(f"elap_player_stats: {df_elap_stats.shape}")

df_elap_schedules = pd.read_csv(f'../{RAW_ELAP_DIR}/team_schedules_2010_2020.csv', index_col=0)
print(f"elap_team_schedules: {df_elap_schedules.shape}")

Loading elap733 data...
missed_games: (11531, 5)
injury_transactions: (14135, 5)
elap_player_stats: (19786, 31)
elap_team_schedules: (28240, 8)


In [3]:
# Load nba_api player stats (combine all years)
print("\nLoading nba_api player stats...")
print("="*60)

player_stats_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'../{RAW_NBA_API_DIR}/player_stats_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        player_stats_dfs.append(df)
        print(f"  player_stats_{season}: {df.shape[0]} players")
    else:
        print(f"  player_stats_{season}: NOT FOUND")

print(f"\nTotal files loaded: {len(player_stats_dfs)}")


Loading nba_api player stats...
  player_stats_2010: 452 players
  player_stats_2011: 478 players
  player_stats_2012: 469 players
  player_stats_2013: 482 players
  player_stats_2014: 492 players
  player_stats_2015: 476 players
  player_stats_2016: 486 players
  player_stats_2017: 540 players
  player_stats_2018: 530 players
  player_stats_2019: 529 players

Total files loaded: 10


In [4]:
# Load nba_api player bio (combine all years)
print("Loading nba_api player bio...")
print("="*60)

player_bio_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'../{RAW_NBA_API_DIR}/player_bio_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        player_bio_dfs.append(df)
        print(f"  player_bio_{season}: {df.shape[0]} players")
    else:
        print(f"  player_bio_{season}: NOT FOUND")

print(f"\nTotal files loaded: {len(player_bio_dfs)}")

Loading nba_api player bio...
  player_bio_2010: 452 players
  player_bio_2011: 478 players
  player_bio_2012: 469 players
  player_bio_2013: 482 players
  player_bio_2014: 492 players
  player_bio_2015: 476 players
  player_bio_2016: 486 players
  player_bio_2017: 540 players
  player_bio_2018: 530 players
  player_bio_2019: 529 players

Total files loaded: 10


In [5]:
# Load nba_api tracking stats (2013+ only)
print("Loading nba_api tracking stats (2013+)...")
print("="*60)

tracking_stats_dfs = []
for season in range(TRACKING_DATA_START, LAST_SEASON + 1):
    filepath = f'../{RAW_NBA_API_DIR}/tracking_stats_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        tracking_stats_dfs.append(df)
        print(f"  tracking_stats_{season}: {df.shape[0]} players")
    else:
        print(f"  tracking_stats_{season}: NOT FOUND")

print(f"\nTotal files loaded: {len(tracking_stats_dfs)}")

Loading nba_api tracking stats (2013+)...
  tracking_stats_2013: 482 players


  tracking_stats_2014: 492 players
  tracking_stats_2015: 476 players
  tracking_stats_2016: 486 players
  tracking_stats_2017: 540 players
  tracking_stats_2018: 530 players
  tracking_stats_2019: 529 players

Total files loaded: 7


In [6]:
# Load nba_api team schedules
print("Loading nba_api team schedules...")
print("="*60)

team_schedules_dfs = []
for season in range(FIRST_SEASON, LAST_SEASON + 1):
    filepath = f'../{RAW_NBA_API_DIR}/team_schedules_{season}.csv'
    if os.path.exists(filepath):
        df = pd.read_csv(filepath)
        team_schedules_dfs.append(df)
        print(f"  team_schedules_{season}: {df.shape[0]} games")
    else:
        print(f"  team_schedules_{season}: NOT FOUND")

print(f"\nTotal files loaded: {len(team_schedules_dfs)}")

Loading nba_api team schedules...
  team_schedules_2010: 2460 games
  team_schedules_2011: 1980 games
  team_schedules_2012: 2460 games


  team_schedules_2013: 2460 games
  team_schedules_2014: 2460 games
  team_schedules_2015: 2460 games
  team_schedules_2016: 2460 games
  team_schedules_2017: 2460 games
  team_schedules_2018: 2460 games
  team_schedules_2019: 2118 games

Total files loaded: 10


---
## Section 2: Clean elap733 Injury Data (CRITICAL)

The `missed_games_2010_2019.csv` has one row per injury event/game missed.

**Key columns:**
- `Relinquished`: Player who missed the game (on IL)
- `Acquired`: Player returning from injury
- `Date`: Date of the event
- `Notes`: Injury description

**Goal:** Aggregate to player-season level with:
- Total games missed
- Number of injury events
- Injury types (parsed from Notes)

In [7]:
# Examine the missed games data
print("Missed Games Data Structure:")
print(df_missed_games_raw.head(10))
print(f"\nColumns: {list(df_missed_games_raw.columns)}")
print(f"\nDate range: {df_missed_games_raw['Date'].min()} to {df_missed_games_raw['Date'].max()}")

Missed Games Data Structure:
         Date       Team Acquired       Relinquished                                              Notes
0  2010-08-03   Clippers      NaN        Craig Smith  arthroscopic surgery on right knee (out indefi...
1  2010-08-13  Mavericks      NaN  Rodrigue Beaubois  surgery on left foot to repair broken fifth me...
2  2010-08-14   Warriors      NaN          Ekpe Udoh           surgery on left wrist (out indefinitely)
3  2010-09-21    Raptors      NaN       Ed Davis (a)  arthroscopic surgery on right kene to repair t...
4  2010-09-21    Thunder      NaN       Nenad Krstic  surgery on right hand to repair broken finger ...
5  2010-09-24    Bobcats      NaN        Kwame Brown             sprained left ankle (out indefinitely)
6  2010-09-27     Knicks      NaN         Eddy Curry        strained right hamstring (out indefinitely)
7  2010-09-27    Rockets      NaN        Brad Miller                          sprained left ankle (DTD)
8  2010-09-29      Magic      NaN  

In [8]:
def parse_nba_season(date_str):
    """
    Convert a date string to NBA season string.
    NBA season runs Oct-Jun, so:
    - Oct-Dec 2015 -> '2015-16'
    - Jan-Jun 2016 -> '2015-16'
    - Jul-Sep = offseason (return None or previous season)
    """
    try:
        date = pd.to_datetime(date_str)
        year = date.year
        month = date.month
        
        if month >= 10:  # Oct-Dec: first year of season
            return f"{year}-{str(year + 1)[-2:]}"
        elif month <= 6:  # Jan-Jun: second year of season
            return f"{year - 1}-{str(year)[-2:]}"
        else:  # Jul-Sep: offseason, assign to upcoming season
            return f"{year}-{str(year + 1)[-2:]}"
    except:
        return None

def get_season_start_year(season_str):
    """Extract the start year from season string like '2015-16' -> 2015"""
    if season_str and '-' in str(season_str):
        return int(season_str.split('-')[0])
    return None

# Test
print("Season parsing tests:")
print(f"  2015-10-15 -> {parse_nba_season('2015-10-15')}")
print(f"  2016-01-15 -> {parse_nba_season('2016-01-15')}")
print(f"  2016-07-15 -> {parse_nba_season('2016-07-15')}")

Season parsing tests:
  2015-10-15 -> 2015-16
  2016-01-15 -> 2015-16
  2016-07-15 -> 2016-17


NBA seasons span two calendar years (e.g., Oct 2015 – Jun 2016 = "2015-16"). July–September events are assigned to the upcoming season.

In [9]:
def extract_injury_type(notes):
    """
    Extract injury type/body part from the Notes column.
    Returns a list of injury keywords found.
    """
    if pd.isna(notes):
        return []
    
    notes_lower = str(notes).lower()
    
    # Body parts to look for
    body_parts = [
        'knee', 'ankle', 'foot', 'achilles', 'hamstring', 'groin',
        'back', 'shoulder', 'wrist', 'hand', 'finger', 'hip',
        'quad', 'calf', 'thigh', 'leg', 'arm', 'elbow',
        'neck', 'head', 'concussion', 'eye', 'nose',
        'toe', 'heel', 'shin', 'rib', 'abdomen', 'abdominal'
    ]
    
    # Injury types
    injury_types = [
        'sprain', 'strain', 'tear', 'fracture', 'broken',
        'surgery', 'sore', 'soreness', 'inflammation',
        'contusion', 'bruise', 'tendinitis', 'tendonitis'
    ]
    
    found = []
    for part in body_parts:
        if part in notes_lower:
            found.append(part)
    
    for inj in injury_types:
        if inj in notes_lower:
            found.append(inj)
    
    return found

def is_rest_day(notes):
    """Check if the entry is rest/load management rather than injury."""
    if pd.isna(notes):
        return False
    notes_lower = str(notes).lower()
    rest_keywords = ['rest', 'load management', 'personal reasons', 'not with team']
    return any(kw in notes_lower for kw in rest_keywords)

# Test
print("Injury extraction tests:")
print(f"  'sprained left ankle (out indefinitely)' -> {extract_injury_type('sprained left ankle (out indefinitely)')}")
print(f"  'torn ACL in right knee' -> {extract_injury_type('torn ACL in right knee')}")
print(f"  'rest (DNP)' -> is_rest: {is_rest_day('rest (DNP)')}")

Injury extraction tests:
  'sprained left ankle (out indefinitely)' -> ['ankle', 'sprain']
  'torn ACL in right knee' -> ['knee']
  'rest (DNP)' -> is_rest: True


In [10]:
def clean_player_name(name):
    """Standardize player name for matching."""
    if pd.isna(name):
        return None
    
    name = str(name).strip()
    
    # Remove common suffixes in parentheses like "(a)" for "acquired"
    name = re.sub(r'\s*\([a-z]\)\s*$', '', name)
    
    # Handle name changes (e.g., "Jeff Pendergraph / Jeff Ayres")
    if '/' in name:
        name = name.split('/')[0].strip()
    
    # Standardize case
    name = ' '.join(word.capitalize() for word in name.split())
    
    return name

# Test
print("Name cleaning tests:")
print(f"  'Ed Davis (a)' -> '{clean_player_name('Ed Davis (a)')}'")
print(f"  'Jeff Pendergraph / Jeff Ayres' -> '{clean_player_name('Jeff Pendergraph / Jeff Ayres')}'")

Name cleaning tests:
  'Ed Davis (a)' -> 'Ed Davis'
  'Jeff Pendergraph / Jeff Ayres' -> 'Jeff Pendergraph'


Now we apply these helper functions to clean the raw injury data — standardizing names, parsing seasons, and separating true injuries from rest days.

In [11]:
# Process missed games data
print("Processing missed games data...")

# Filter to only rows where a player is "Relinquished" (missed a game)
df_missed = df_missed_games_raw[df_missed_games_raw['Relinquished'].notna()].copy()
print(f"Rows with player relinquished: {len(df_missed)}")

# Clean player names
df_missed['player_name'] = df_missed['Relinquished'].apply(clean_player_name)

# Filter out coaches/executives (these appear in transaction data but aren't players)
coaches_removed = df_missed['player_name'].isin(EXCLUDE_FROM_INJURY_DATA).sum()
df_missed = df_missed[~df_missed['player_name'].isin(EXCLUDE_FROM_INJURY_DATA)]
print(f"Removed {coaches_removed} entries for coaches/staff")

# Parse season
df_missed['season'] = df_missed['Date'].apply(parse_nba_season)
df_missed['season_start_year'] = df_missed['season'].apply(get_season_start_year)

# Extract injury info
df_missed['injury_types'] = df_missed['Notes'].apply(extract_injury_type)
df_missed['is_rest'] = df_missed['Notes'].apply(is_rest_day)

print(f"\nSample processed data:")
df_missed[['Date', 'player_name', 'season', 'Notes', 'injury_types', 'is_rest']].head(10)

Processing missed games data...
Rows with player relinquished: 9328


Removed 22 entries for coaches/staff



Sample processed data:


,Date,player_name,season,Notes,injury_types,is_rest
0,2010-08-03,Craig Smith,2010-11,arthroscopic surgery on right knee (out indefi...,"[knee, surgery]",False
1,2010-08-13,Rodrigue Beaubois,2010-11,surgery on left foot to repair broken fifth me...,"[foot, broken, surgery]",False
2,2010-08-14,Ekpe Udoh,2010-11,surgery on left wrist (out indefinitely),"[wrist, surgery]",False
3,2010-09-21,Ed Davis,2010-11,arthroscopic surgery on right kene to repair t...,[surgery],False
4,2010-09-21,Nenad Krstic,2010-11,surgery on right hand to repair broken finger ...,"[hand, finger, broken, surgery]",False
5,2010-09-24,Kwame Brown,2010-11,sprained left ankle (out indefinitely),"[ankle, sprain]",False
6,2010-09-27,Eddy Curry,2010-11,strained right hamstring (out indefinitely),"[hamstring, strain]",False
7,2010-09-27,Brad Miller,2010-11,sprained left ankle (DTD),"[ankle, sprain]",False
8,2010-09-29,Jason Williams,2010-11,arthroscopic surgery on left knee (out indefin...,"[knee, surgery]",False
9,2010-10-03,Carlos Boozer,2010-11,fractured bone in right pinky finger (out inde...,"[finger, fracture]",False


In [12]:
# Check rest vs injury breakdown
print("Rest vs Injury breakdown:")
print(df_missed['is_rest'].value_counts())
print(f"\nRest percentage: {df_missed['is_rest'].mean()*100:.1f}%")

Rest vs Injury breakdown:
is_rest
False    8355
True      951
Name: count, dtype: int64

Rest percentage: 10.2%


In [13]:
# Decision: EXCLUDE pure rest days, INCLUDE injuries only
# This is a modeling decision - we're predicting injury risk, not rest decisions

df_injuries_only = df_missed[~df_missed['is_rest']].copy()
print(f"After excluding rest days: {len(df_injuries_only)} injury events")

After excluding rest days: 8355 injury events


Rest days (10.2% of entries) are load management decisions, not injuries. Since we're predicting injury risk, we exclude them.

In [14]:
# Filter to our season range (2010-2019)
df_injuries_only = df_injuries_only[
    (df_injuries_only['season_start_year'] >= FIRST_SEASON) & 
    (df_injuries_only['season_start_year'] <= LAST_SEASON)
].copy()

print(f"After filtering to {FIRST_SEASON}-{LAST_SEASON}: {len(df_injuries_only)} injury events")
print(f"\nSeasons covered:")
print(df_injuries_only['season'].value_counts().sort_index())

After filtering to 2010-2019: 8355 injury events

Seasons covered:
season
2010-11     824
2011-12    1310
2012-13    1227
2013-14    1742
2014-15     615
2015-16     722
2016-17     731
2017-18     619
2018-19     564
2019-20       1
Name: count, dtype: int64


In [15]:
# Aggregate to player-season level
print("Aggregating to player-season level...")

def aggregate_injuries(group):
    """Aggregate injury data for a player-season."""
    # Flatten all injury types
    all_injuries = []
    for inj_list in group['injury_types']:
        all_injuries.extend(inj_list)
    
    # Get unique injury types
    unique_injuries = list(set(all_injuries))
    
    return pd.Series({
        'games_missed': len(group),  # Each row = one game missed
        'injury_events': group['Notes'].nunique(),  # Unique injury descriptions
        'injury_types': unique_injuries,
        'teams': list(group['Team'].unique())
    })

df_injury_by_player_season = df_injuries_only.groupby(
    ['player_name', 'season', 'season_start_year']
).apply(aggregate_injuries, include_groups=False).reset_index()

print(f"\nPlayer-season injury records: {len(df_injury_by_player_season)}")
df_injury_by_player_season.head(10)

Aggregating to player-season level...



Player-season injury records: 2623


/var/folders/tt/q6yxx0qn7dq_jbphcnvcmtfr0000gn/T/ipykernel_91969/351415098.py:23: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ).apply(aggregate_injuries).reset_index()


,player_name,season,season_start_year,games_missed,injury_events,injury_types,teams
0,(william) Tony Parker,2010-11,2010,2,2,"[bruise, calf, strain]",[Spurs]
1,(william) Tony Parker,2011-12,2011,1,1,"[sore, hamstring]",[Spurs]
2,(william) Tony Parker,2012-13,2012,5,5,"[ankle, sore, neck, knee, sprain]",[Spurs]
3,(william) Tony Parker,2013-14,2013,2,2,"[bruise, shin, achilles]",[Spurs]
4,(william) Tony Parker,2014-15,2014,3,3,"[rib, bruise, strain, hamstring]",[Spurs]
5,(william) Tony Parker,2015-16,2015,3,3,"[bruise, ankle, hip, toe]",[Spurs]
6,(william) Tony Parker,2016-17,2016,6,5,"[sore, leg, back, foot, knee, quad]",[Spurs]
7,(william) Tony Parker,2017-18,2017,2,2,"[ankle, sprain, back]",[Spurs]
8,(william) Tony Parker,2018-19,2018,1,1,[rib],[Hornets]
9,A.j. Price,2011-12,2011,2,1,[],[Pacers]


In [16]:
# Summary statistics
print("Injury Summary Statistics:")
print(f"\nGames missed per player-season:")
print(df_injury_by_player_season['games_missed'].describe())

print(f"\nTop 10 most injury-prone player-seasons:")
df_injury_by_player_season.nlargest(10, 'games_missed')[['player_name', 'season', 'games_missed', 'injury_types']]

Injury Summary Statistics:

Games missed per player-season:
count    2623.000000
mean        3.185284
std         3.750411
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max        40.000000
Name: games_missed, dtype: float64

Top 10 most injury-prone player-seasons:


,player_name,season,games_missed,injury_types
1472,Kevin Love,2012-13,40,"[hand, surgery, eye, finger, fracture, knee]"
951,Greg Smith,2013-14,39,"[sore, hip, surgery, knee, sprain]"
1151,Jason Smith,2013-14,39,"[bruise, knee]"
1883,Nate Robinson,2013-14,37,"[knee, surgery]"
1323,Jordan Farmar,2013-14,31,"[groin, sore, strain, hamstring]"
781,Ekpe Udoh,2013-14,28,"[knee, ankle, sprain, surgery]"
2315,Steve Blake,2013-14,27,[elbow]
2541,Vitor Faverani,2013-14,27,[knee]
2418,Tony Allen,2013-14,26,"[wrist, hand, hip, sprain]"
800,Emeka Okafor,2011-12,25,"[knee, sore]"


In [17]:
# Save injury history
output_path = f'../{PROCESSED_DIR}/injury_history_by_player_season.csv'
df_injury_by_player_season.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_injury_by_player_season.shape}")

Saved: ../data/processed/injury_history_by_player_season.csv
Shape: (2623, 7)


### Section 2 Summary

**Decisions made:**
- **Excluded pure rest days** - We're predicting injury risk, not load management decisions
- **Counted each row as one game missed** - The data has one row per DNP event
- **Aggregated to player-season level** with games missed, injury count, and injury types

**Output:** `injury_history_by_player_season.csv` with columns:
- `player_name`, `season`, `season_start_year`
- `games_missed` (total games missed due to injury)
- `injury_events` (number of distinct injury incidents)
- `injury_types` (list of body parts/injury types)

---
## Section 3: Clean nba_api Player Stats

In [18]:
# Combine all player stats files
print("Combining player stats files...")

if player_stats_dfs:
    df_player_stats = pd.concat(player_stats_dfs, ignore_index=True)
    print(f"Combined shape: {df_player_stats.shape}")
    print(f"\nColumns: {list(df_player_stats.columns)}")
else:
    print("No player stats files loaded!")
    df_player_stats = pd.DataFrame()

Combining player stats files...
Combined shape: (4934, 69)

Columns: ['PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'GP', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK', 'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2', 'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK', 'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK', 'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK', 'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK', 'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK', 'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK', 'WNBA_FANTASY_PTS_RANK', 'TEAM_COUNT', 'SEASON', 'SEASON_START_YEAR']


Players traded mid-season could appear twice for the same season. We check for and resolve duplicates.

In [19]:
# Check for duplicates
if not df_player_stats.empty:
    dupes = df_player_stats.duplicated(subset=['PLAYER_ID', 'SEASON'], keep=False)
    print(f"Duplicate player-season records: {dupes.sum()}")
    
    if dupes.sum() > 0:
        print("\nSample duplicates:")
        print(df_player_stats[dupes][['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ABBREVIATION', 'SEASON']].head(10))
        
        # Players traded mid-season appear twice - keep the row with more games
        print("\nKeeping row with most games played for each player-season...")
        df_player_stats = df_player_stats.sort_values('GP', ascending=False).drop_duplicates(
            subset=['PLAYER_ID', 'SEASON'], keep='first'
        )
        print(f"Shape after deduplication: {df_player_stats.shape}")

Duplicate player-season records: 0


In [20]:
# Standardize column names (lowercase with underscores)
df_player_stats.columns = df_player_stats.columns.str.lower().str.replace(' ', '_')

# Clean player names
df_player_stats['player_name_clean'] = df_player_stats['player_name'].apply(clean_player_name)

print(f"Columns: {list(df_player_stats.columns)}")
df_player_stats.head(3)

Columns: ['player_id', 'player_name', 'nickname', 'team_id', 'team_abbreviation', 'age', 'gp', 'w', 'l', 'w_pct', 'min', 'fgm', 'fga', 'fg_pct', 'fg3m', 'fg3a', 'fg3_pct', 'ftm', 'fta', 'ft_pct', 'oreb', 'dreb', 'reb', 'ast', 'tov', 'stl', 'blk', 'blka', 'pf', 'pfd', 'pts', 'plus_minus', 'nba_fantasy_pts', 'dd2', 'td3', 'wnba_fantasy_pts', 'gp_rank', 'w_rank', 'l_rank', 'w_pct_rank', 'min_rank', 'fgm_rank', 'fga_rank', 'fg_pct_rank', 'fg3m_rank', 'fg3a_rank', 'fg3_pct_rank', 'ftm_rank', 'fta_rank', 'ft_pct_rank', 'oreb_rank', 'dreb_rank', 'reb_rank', 'ast_rank', 'tov_rank', 'stl_rank', 'blk_rank', 'blka_rank', 'pf_rank', 'pfd_rank', 'pts_rank', 'plus_minus_rank', 'nba_fantasy_pts_rank', 'dd2_rank', 'td3_rank', 'wnba_fantasy_pts_rank', 'team_count', 'season', 'season_start_year', 'player_name_clean']


,player_id,player_name,nickname,team_id,team_abbreviation,age,gp,w,l,w_pct,min,fgm,fga,fg_pct,fg3m,fg3a,fg3_pct,ftm,fta,ft_pct,oreb,dreb,reb,ast,tov,...,fg3a_rank,fg3_pct_rank,ftm_rank,fta_rank,ft_pct_rank,oreb_rank,dreb_rank,reb_rank,ast_rank,tov_rank,stl_rank,blk_rank,blka_rank,pf_rank,pfd_rank,pts_rank,plus_minus_rank,nba_fantasy_pts_rank,dd2_rank,td3_rank,wnba_fantasy_pts_rank,team_count,season,season_start_year,player_name_clean
0,201985,AJ Price,AJ,1610612754,IND,24.0,50,22,28,0.440,15.9,2.3,6.4,0.356,0.8,3.0,0.275,1.1,1.6,0.667,0.3,1.1,1.4,2.2,1.1,...,97,241,234,212,316,348,369,375,121,203,208,421,308,351,198,240,161,274,224,27,264,1,2010-11,2010,Aj Price
1,201166,Aaron Brooks,Aaron,1610612756,PHX,26.0,59,26,33,0.441,21.8,3.7,9.9,0.375,1.2,4.0,0.297,2.1,2.4,0.886,0.3,1.0,1.3,3.9,1.7,...,55,227,104,138,30,344,388,385,44,99,209,392,62,205,131,132,377,179,154,27,164,2,2010-11,2010,Aaron Brooks
2,201189,Aaron Gray,Aaron,1610612740,NOH,26.0,41,21,20,0.512,12.9,1.4,2.4,0.566,0.0,0.0,0.000,0.4,0.8,0.500,1.4,2.7,4.2,0.4,0.8,...,383,309,367,333,398,98,169,133,376,290,368,223,270,121,348,362,145,326,154,27,332,1,2010-11,2010,Aaron Gray


In [21]:
# Check for missing values in key columns
key_cols = ['player_id', 'player_name', 'season', 'gp', 'min', 'pts']
print("Missing values in key columns:")
print(df_player_stats[key_cols].isnull().sum())

Missing values in key columns:
player_id      0
player_name    0
season         0
gp             0
min            0
pts            0
dtype: int64


In [22]:
# Save combined player stats
output_path = f'../{PROCESSED_DIR}/player_stats_combined.csv'
df_player_stats.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_player_stats.shape}")

Saved: ../data/processed/player_stats_combined.csv
Shape: (4934, 70)


---
## Section 4: Clean nba_api Player Bio

In [23]:
# Combine all player bio files
print("Combining player bio files...")

if player_bio_dfs:
    df_player_bio = pd.concat(player_bio_dfs, ignore_index=True)
    print(f"Combined shape: {df_player_bio.shape}")
    print(f"\nColumns: {list(df_player_bio.columns)}")
else:
    print("No player bio files loaded!")
    df_player_bio = pd.DataFrame()

Combining player bio files...
Combined shape: (4934, 25)

Columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'AGE', 'PLAYER_HEIGHT', 'PLAYER_HEIGHT_INCHES', 'PLAYER_WEIGHT', 'COLLEGE', 'COUNTRY', 'DRAFT_YEAR', 'DRAFT_ROUND', 'DRAFT_NUMBER', 'GP', 'PTS', 'REB', 'AST', 'NET_RATING', 'OREB_PCT', 'DREB_PCT', 'USG_PCT', 'TS_PCT', 'AST_PCT', 'SEASON', 'SEASON_START_YEAR']


In [24]:
if not df_player_bio.empty:
    # Standardize column names
    df_player_bio.columns = df_player_bio.columns.str.lower().str.replace(' ', '_')
    
    # Check for height format
    print("Sample heights:")
    print(df_player_bio['player_height'].head(10))
    
    # If height is in "6-10" format, convert to inches
    def height_to_inches(height_str):
        """Convert height like '6-10' to inches (82)."""
        if pd.isna(height_str):
            return None
        try:
            if '-' in str(height_str):
                feet, inches = str(height_str).split('-')
                return int(feet) * 12 + int(inches)
            else:
                return float(height_str)
        except:
            return None
    
    df_player_bio['height_inches'] = df_player_bio['player_height'].apply(height_to_inches)
    print(f"\nHeight conversion sample:")
    print(df_player_bio[['player_height', 'height_inches']].head(5))

Sample heights:
0     6-2
1     6-0
2     7-0
3     6-3
4     6-9
5    6-10
6    6-10
7     6-8
8     6-9
9     7-2
Name: player_height, dtype: object

Height conversion sample:
  player_height  height_inches
0           6-2           74.0
1           6-0           72.0
2           7-0           84.0
3           6-3           75.0
4           6-9           81.0


Height is stored as "6-10" format — converting to inches for numerical analysis.

In [25]:
if not df_player_bio.empty:
    # Clean player names
    df_player_bio['player_name_clean'] = df_player_bio['player_name'].apply(clean_player_name)
    
    # Handle duplicates (same player in multiple seasons)
    dupes = df_player_bio.duplicated(subset=['player_id', 'season'], keep=False)
    print(f"Duplicate player-season records: {dupes.sum()}")
    
    if dupes.sum() > 0:
        df_player_bio = df_player_bio.drop_duplicates(subset=['player_id', 'season'], keep='first')
        print(f"Shape after deduplication: {df_player_bio.shape}")
    
    df_player_bio.head(3)

Duplicate player-season records: 0


In [26]:
# Save combined player bio
if not df_player_bio.empty:
    output_path = f'../{PROCESSED_DIR}/player_bio_combined.csv'
    df_player_bio.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    print(f"Shape: {df_player_bio.shape}")

Saved: ../data/processed/player_bio_combined.csv
Shape: (4934, 27)


---
## Section 5: Clean Tracking Data (2013+ only)

In [27]:
# Combine tracking stats
print("Combining tracking stats files...")

if tracking_stats_dfs:
    df_tracking = pd.concat(tracking_stats_dfs, ignore_index=True)
    print(f"Combined shape: {df_tracking.shape}")
    print(f"\nColumns: {list(df_tracking.columns)}")
else:
    print("No tracking stats files loaded!")
    df_tracking = pd.DataFrame()

Combining tracking stats files...
Combined shape: (3535, 18)

Columns: ['PLAYER_ID', 'PLAYER_NAME', 'TEAM_ID', 'TEAM_ABBREVIATION', 'GP', 'W', 'L', 'MIN', 'MIN1', 'DIST_FEET', 'DIST_MILES', 'DIST_MILES_OFF', 'DIST_MILES_DEF', 'AVG_SPEED', 'AVG_SPEED_OFF', 'AVG_SPEED_DEF', 'SEASON', 'SEASON_START_YEAR']


In [28]:
if not df_tracking.empty:
    # Standardize column names
    df_tracking.columns = df_tracking.columns.str.lower().str.replace(' ', '_')
    
    # Clean player names
    df_tracking['player_name_clean'] = df_tracking['player_name'].apply(clean_player_name)
    
    # Handle duplicates
    dupes = df_tracking.duplicated(subset=['player_id', 'season'], keep=False)
    print(f"Duplicate player-season records: {dupes.sum()}")
    
    if dupes.sum() > 0:
        df_tracking = df_tracking.drop_duplicates(subset=['player_id', 'season'], keep='first')
    
    print(f"\nSeasons in tracking data:")
    print(df_tracking['season'].value_counts().sort_index())
    
    df_tracking.head(3)

Duplicate player-season records: 0

Seasons in tracking data:
season
2013-14    482
2014-15    492
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
Name: count, dtype: int64


In [29]:
# Save combined tracking data
if not df_tracking.empty:
    output_path = f'../{PROCESSED_DIR}/tracking_stats_combined.csv'
    df_tracking.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    print(f"Shape: {df_tracking.shape}")

Saved: ../data/processed/tracking_stats_combined.csv
Shape: (3535, 19)


---
## Section 6: Calculate Back-to-Back Games

In [30]:
# Combine team schedules
print("Combining team schedule files...")

if team_schedules_dfs:
    df_schedules = pd.concat(team_schedules_dfs, ignore_index=True)
    print(f"Combined shape: {df_schedules.shape}")
    print(f"\nColumns: {list(df_schedules.columns)}")
else:
    print("No team schedule files loaded!")
    df_schedules = pd.DataFrame()

Combining team schedule files...
Combined shape: (23778, 30)

Columns: ['Team_ID', 'Game_ID', 'GAME_DATE', 'MATCHUP', 'WL', 'W', 'L', 'W_PCT', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'TEAM_ABBREVIATION', 'SEASON', 'SEASON_START_YEAR']


In [31]:
if not df_schedules.empty:
    # Standardize column names
    df_schedules.columns = df_schedules.columns.str.lower().str.replace(' ', '_')
    
    # Parse game date
    df_schedules['game_date'] = pd.to_datetime(df_schedules['game_date'])
    
    # Sort by team and date
    df_schedules = df_schedules.sort_values(['team_id', 'game_date'])
    
    # Calculate days since last game for each team
    df_schedules['days_rest'] = df_schedules.groupby('team_id')['game_date'].diff().dt.days
    
    # Flag back-to-back games (played day after previous game)
    df_schedules['is_back_to_back'] = (df_schedules['days_rest'] == 1).astype(int)
    
    print(f"\nBack-to-back game frequency:")
    print(df_schedules['is_back_to_back'].value_counts())
    print(f"\nB2B percentage: {df_schedules['is_back_to_back'].mean()*100:.1f}%")


Back-to-back game frequency:
is_back_to_back
0    18677
1     5101
Name: count, dtype: int64

B2B percentage: 21.5%


/var/folders/tt/q6yxx0qn7dq_jbphcnvcmtfr0000gn/T/ipykernel_91969/116385327.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_schedules['game_date'] = pd.to_datetime(df_schedules['game_date'])


In [32]:
if not df_schedules.empty:
    # Aggregate back-to-backs by team-season
    df_b2b_summary = df_schedules.groupby(['team_id', 'team_abbreviation', 'season']).agg({
        'is_back_to_back': 'sum',
        'game_date': 'count'
    }).reset_index()
    
    df_b2b_summary.columns = ['team_id', 'team_abbreviation', 'season', 'b2b_games', 'total_games']
    df_b2b_summary['b2b_percentage'] = df_b2b_summary['b2b_games'] / df_b2b_summary['total_games'] * 100
    
    print("Back-to-back summary by team-season:")
    df_b2b_summary.head(10)

Back-to-back summary by team-season:


About 21.5% of all NBA games are back-to-backs. This schedule data will be a key feature for injury prediction.

In [33]:
# Save back-to-back data
if not df_schedules.empty:
    # Save detailed game-level data
    output_path = f'../{PROCESSED_DIR}/team_schedules_with_b2b.csv'
    df_schedules.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")
    
    # Save summary
    output_path = f'../{PROCESSED_DIR}/team_back_to_backs.csv'
    df_b2b_summary.to_csv(output_path, index=False)
    print(f"Saved: {output_path}")

Saved: ../data/processed/team_schedules_with_b2b.csv
Saved: ../data/processed/team_back_to_backs.csv


---
## Section 7: Create Player ID Mapping (CRITICAL)

**Challenge:** elap733 injury data uses player names only, while nba_api uses PLAYER_ID.

We need to create a mapping between:
- Player names (from injury data)
- NBA Player IDs (from nba_api)

This requires fuzzy matching for name variations.

In [34]:
# Get unique players from each source
print("Unique players in each dataset:")

# From injury data - apply manual mappings
injury_players_raw = set(df_injury_by_player_season['player_name'].unique())
print(f"  Injury data (raw): {len(injury_players_raw)} unique players")

# Apply manual name mappings to injury player names
injury_players_mapped = {}
for player in injury_players_raw:
    mapped_name = MANUAL_NAME_MAPPINGS.get(player, player)
    injury_players_mapped[player] = mapped_name
    if player != mapped_name:
        print(f"    Mapped: '{player}' -> '{mapped_name}'")

injury_players = set(injury_players_mapped.values())
print(f"  Injury data (after mapping): {len(injury_players)} unique players")

# From nba_api stats
if not df_player_stats.empty:
    nba_players = df_player_stats[['player_id', 'player_name', 'player_name_clean']].drop_duplicates()
    print(f"  NBA API stats: {len(nba_players)} unique players")
else:
    nba_players = pd.DataFrame()

Unique players in each dataset:
  Injury data (raw): 850 unique players
    Mapped: 'Frank Mason' -> 'Frank Mason III'
    Mapped: 'Nikola Vucevic' -> 'Nikola Vučević'
    Mapped: 'Hidayet Turkoglu' -> 'Hedo Turkoglu'
    Mapped: 'Predrag Stojakovic' -> 'Peja Stojakovic'
    Mapped: 'Patrick Mills' -> 'Patty Mills'
    Mapped: 'Guillermo Hernangomez' -> 'Willy Hernangomez'
    Mapped: 'Sviatoslav Mykhailiuk' -> 'Svi Mykhailiuk'
    Mapped: 'Dario Saric' -> 'Dario Šarić'
    Mapped: 'Bogdan Bogdanovic' -> 'Bogdan Bogdanović'
    Mapped: 'Luc Richard Mbah A Moute' -> 'Luc Mbah A Moute'
    Mapped: 'Ersan Ilyasova' -> 'Ersan İlyasova'
    Mapped: 'Donatas Motiejunas' -> 'Donatas Motiejūnas'
    Mapped: 'Jonas Valanciunas' -> 'Jonas Valančiūnas'
    Mapped: 'Luka Doncic' -> 'Luka Dončić'
    Mapped: 'Tony Wroten Jr.' -> 'Tony Wroten'
    Mapped: 'Enes Kanter' -> 'Enes Freedom'
    Mapped: 'Glen Rice Jr.' -> 'Glen Rice'
    Mapped: 'Maximilian Kleber' -> 'Maxi Kleber'
    Mapped: 'Mohamed B

Exact name matching won't catch all players due to special characters (Nikola Jokić), nicknames (Bam Adebayo), and suffixes (Jr./III). We use fuzzy matching as a fallback.

In [35]:
def fuzzy_match_score(name1, name2):
    """Calculate similarity score between two names."""
    if not name1 or not name2:
        return 0
    return SequenceMatcher(None, name1.lower(), name2.lower()).ratio()

def find_best_match(name, candidates, threshold=0.85):
    """
    Find the best matching name from candidates.
    Returns (matched_name, score, player_id) or (None, 0, None) if no match.
    """
    best_match = None
    best_score = 0
    best_id = None
    
    for _, row in candidates.iterrows():
        candidate_name = row['player_name_clean']
        score = fuzzy_match_score(name, candidate_name)
        
        if score > best_score:
            best_score = score
            best_match = candidate_name
            best_id = row['player_id']
    
    if best_score >= threshold:
        return best_match, best_score, best_id
    return None, best_score, None

# Test fuzzy matching
print("Fuzzy matching tests:")
print(f"  'LeBron James' vs 'Lebron James': {fuzzy_match_score('LeBron James', 'Lebron James'):.3f}")
print(f"  'Stephen Curry' vs 'Steph Curry': {fuzzy_match_score('Stephen Curry', 'Steph Curry'):.3f}")

Fuzzy matching tests:
  'LeBron James' vs 'Lebron James': 1.000
  'Stephen Curry' vs 'Steph Curry': 0.917


In [36]:
# Create player ID mapping
print("Creating player ID mapping...")
print("This may take a few minutes for fuzzy matching...")

mapping_results = []
unmatched = []

# Process each original injury player name
for i, (original_name, mapped_name) in enumerate(injury_players_mapped.items()):
    if i % 100 == 0:
        print(f"  Processing {i}/{len(injury_players_mapped)}...")
    
    # First try exact match with mapped name
    exact_match = nba_players[nba_players['player_name_clean'] == mapped_name]
    
    # Also try matching against player_name directly (for special characters)
    if len(exact_match) == 0:
        exact_match = nba_players[nba_players['player_name'] == mapped_name]
    
    if len(exact_match) > 0:
        # Exact match found
        match_type = 'manual' if original_name != mapped_name else 'exact'
        mapping_results.append({
            'injury_player_name': original_name,
            'nba_player_name': exact_match.iloc[0]['player_name'],
            'nba_player_id': exact_match.iloc[0]['player_id'],
            'match_type': match_type,
            'match_score': 1.0
        })
    else:
        # Try fuzzy match
        matched_name, score, player_id = find_best_match(mapped_name, nba_players)
        
        if matched_name:
            mapping_results.append({
                'injury_player_name': original_name,
                'nba_player_name': matched_name,
                'nba_player_id': player_id,
                'match_type': 'fuzzy',
                'match_score': score
            })
        else:
            unmatched.append({
                'injury_player_name': original_name,
                'mapped_name': mapped_name,
                'best_score': score
            })

print(f"\nMatching complete!")
print(f"  Matched: {len(mapping_results)}")
print(f"  Unmatched: {len(unmatched)}")

Creating player ID mapping...
This may take a few minutes for fuzzy matching...
  Processing 0/850...


  Processing 100/850...


  Processing 200/850...
  Processing 300/850...
  Processing 400/850...


  Processing 500/850...


  Processing 600/850...


  Processing 700/850...


  Processing 800/850...

Matching complete!
  Matched: 843
  Unmatched: 7


In [37]:
# Create mapping dataframe
df_player_mapping = pd.DataFrame(mapping_results)

print("Match type breakdown:")
print(df_player_mapping['match_type'].value_counts())

print(f"\nFuzzy matches (review these):")
fuzzy_matches = df_player_mapping[df_player_mapping['match_type'] == 'fuzzy'].sort_values('match_score')
print(fuzzy_matches[['injury_player_name', 'nba_player_name', 'match_score']].head(20))

Match type breakdown:
match_type
exact     776
manual     35
fuzzy      32
Name: count, dtype: int64

Fuzzy matches (review these):
       injury_player_name     nba_player_name  match_score
615          Danuel House    Danuel House Jr.     0.857143
472  Marcus Thornton (t.)     Marcus Thornton     0.857143
189      Roy Devyn Marble        Devyn Marble     0.857143
477          Jimmy Butler    Jimmy Butler Iii     0.857143
433          John Wallace           John Wall     0.857143
94          Marcus Morris   Marcus Morris Sr.     0.866667
287         Wesley Iwundu          Wes Iwundu     0.869565
805            Kevin Knox       Kevin Knox Ii     0.869565
441        Reggie Bullock  Reggie Bullock Jr.     0.875000
718        Monty Williams         Mo Williams     0.880000
15             D.j. White            Dj White     0.888889
676            J.r. Smith            Jr Smith     0.888889
653            A.j. Price            Aj Price     0.888889
379            C.j. Miles            Cj Mi

In [38]:
# Show unmatched players
if unmatched:
    df_unmatched = pd.DataFrame(unmatched)
    print(f"\nUnmatched players ({len(df_unmatched)}):")
    print(df_unmatched.sort_values('best_score', ascending=False))
else:
    print("\nAll players matched successfully!")
    df_unmatched = pd.DataFrame()


Unmatched players (7):
    injury_player_name          mapped_name  best_score
2        Terrico White        Terrico White    0.846154
1       Walter Tavares       Walter Tavares    0.720000
5    Sheldon Mcclellan    Sheldon Mcclellan    0.714286
0  Mike James (lamont)  Mike James (lamont)    0.689655
3       Trevon Bluiett       Trevon Bluiett    0.615385
6      Yakuba Ouattara      Yakuba Ouattara    0.571429
4     Jeff Pendergraph     Jeff Pendergraph    0.538462


Only 7 players remain unmatched — all minor players with minimal games. This 99.2% match rate is sufficient for our analysis.

In [39]:
# Save player mapping
output_path = f'../{PROCESSED_DIR}/player_id_mapping.csv'
df_player_mapping.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Shape: {df_player_mapping.shape}")

# Show match type breakdown
print(f"\nMatch type breakdown:")
print(df_player_mapping['match_type'].value_counts())

# Also save unmatched for manual review
if len(df_unmatched) > 0:
    output_path = f'../{PROCESSED_DIR}/unmatched_players.csv'
    df_unmatched.to_csv(output_path, index=False)
    print(f"\nSaved unmatched players: {output_path}")
else:
    # Remove old unmatched file if all players are now matched
    unmatched_path = f'../{PROCESSED_DIR}/unmatched_players.csv'
    if os.path.exists(unmatched_path):
        os.remove(unmatched_path)
        print(f"\nRemoved old unmatched_players.csv (all players now matched)")

Saved: ../data/processed/player_id_mapping.csv
Shape: (843, 5)

Match type breakdown:
match_type
exact     776
manual     35
fuzzy      32
Name: count, dtype: int64

Saved unmatched players: ../data/processed/unmatched_players.csv


---
## Section 8: Summary & Verification

In [40]:
print("="*60)
print("DATA CLEANING SUMMARY")
print("="*60)

# List all output files
print("\nProcessed files created:")
for f in sorted(os.listdir(f'../{PROCESSED_DIR}')):
    if f.endswith('.csv'):
        filepath = f'../{PROCESSED_DIR}/{f}'
        df = pd.read_csv(filepath)
        size_kb = os.path.getsize(filepath) / 1024
        print(f"  {f}: {df.shape[0]} rows, {df.shape[1]} cols, {size_kb:.1f} KB")

DATA CLEANING SUMMARY

Processed files created:
  injury_history_by_player_season.csv: 2623 rows, 7 cols, 176.8 KB
  player_bio_combined.csv: 4934 rows, 27 cols, 792.3 KB
  player_id_mapping.csv: 843 rows, 5 cols, 37.6 KB
  player_stats_combined.csv: 4934 rows, 70 cols, 1500.1 KB
  team_back_to_backs.csv: 300 rows, 6 cols, 14.0 KB


  team_schedules_with_b2b.csv: 23778 rows, 32 cols, 3298.1 KB
  tracking_stats_combined.csv: 3535 rows, 19 cols, 398.3 KB
  unmatched_players.csv: 7 rows, 3 cols, 0.4 KB


In [41]:
# Verify injury data coverage
print("\nInjury Data Coverage:")
print(f"  Total player-seasons with injuries: {len(df_injury_by_player_season)}")
print(f"  Unique players: {df_injury_by_player_season['player_name'].nunique()}")
print(f"  Seasons covered: {sorted(df_injury_by_player_season['season'].unique())}")


Injury Data Coverage:
  Total player-seasons with injuries: 2623
  Unique players: 850
  Seasons covered: ['2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20']


In [42]:
# Verify player mapping coverage
print("\nPlayer Mapping Coverage:")
total_injury_players = len(injury_players)
matched_players = len(df_player_mapping)
print(f"  Players in injury data: {total_injury_players}")
print(f"  Players matched to NBA ID: {matched_players}")
print(f"  Match rate: {matched_players/total_injury_players*100:.1f}%")


Player Mapping Coverage:
  Players in injury data: 850
  Players matched to NBA ID: 843
  Match rate: 99.2%


In [43]:
# Data quality checks
print("\nData Quality Checks:")

# Check for any seasons with suspiciously low data
if not df_player_stats.empty:
    players_per_season = df_player_stats.groupby('season').size()
    print(f"\nPlayers per season (stats):")
    print(players_per_season)

# Check injury distribution
injuries_per_season = df_injury_by_player_season.groupby('season')['games_missed'].sum()
print(f"\nTotal games missed per season:")
print(injuries_per_season)


Data Quality Checks:

Players per season (stats):
season
2010-11    452
2011-12    478
2012-13    469
2013-14    482
2014-15    492
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
dtype: int64

Total games missed per season:
season
2010-11     824
2011-12    1310
2012-13    1227
2013-14    1742
2014-15     615
2015-16     722
2016-17     731
2017-18     619
2018-19     564
2019-20       1
Name: games_missed, dtype: int64


---
## Summary

### Output Files (`data/processed/`)

| File | Description |
|------|-------------|
| `injury_history_by_player_season.csv` | Games missed per player per season (target variable source) |
| `player_stats_combined.csv` | Combined player stats (2010-2019) |
| `player_bio_combined.csv` | Player demographics (height, weight, age) |
| `tracking_stats_combined.csv` | Speed/distance tracking (2013+ only) |
| `team_back_to_backs.csv` | B2B games by team-season |
| `team_schedules_with_b2b.csv` | Game-level schedule with B2B flags |
| `player_id_mapping.csv` | Name-to-ID mapping (99.2% match rate) |

### Key Decisions

- Excluded rest days (10.2% of entries) — we're predicting injury, not load management
- Excluded coaches/staff from injury data (14 names filtered)
- Used fuzzy matching (threshold ≥0.85) for player names with special characters or nicknames
- 7 unmatched players remain — all minor players with minimal games

All cleaned data is ready for exploration in `03_eda.ipynb`.